# RIGHTWARE SIC | E1-X01 — Ranking treinável experimental
**Missão:** medir baselines e treinar um protótipo RW-R1-X01 em dados sintéticos novos, sem dependências externas ou GPU. **Não é E1_DATA_READY, E1.2/E1.3 oficial, nem avaliação científica independente.**

Kaggle: import notebook → Accelerator None → Internet Off → Run All. O último passo gera `RIGHTWARE_SIC_E1_X01_EVIDENCE.zip`. Não carregar outros dados, chaves, nem notebooks anteriores.

**Contrato congelado:** 48 grupos sintéticos separados por entidade; 28 train / 10 dev / 10 diagnostic. Etiquetas por regras de construção, não anotação humana. Diagnostic é público e sintético, não holdout cego. Comparação primária nDCG@5 de suporte oficial e MRR; métricas negativas, idiomas, scope e latência separadas. Sem escolher configuração pelo diagnostic. Nenhum acesso ao corpus E1.1B ou A0 v0.8.

In [ ]:
import os
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
import json, hashlib, random, re, math, time, platform, zipfile, uuid
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import numpy as np
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import brier_score_loss
import matplotlib.pyplot as plt
from IPython.display import FileLink, display
SEED=20261124
rng=random.Random(SEED)
ROOT=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT=ROOT/'SIC_E1_X01_RESULTS'; OUT.mkdir(exist_ok=True)
ZIP=ROOT/'RIGHTWARE_SIC_E1_X01_EVIDENCE.zip'
for p in OUT.iterdir():
    if p.is_file(): p.unlink()
if ZIP.exists(): ZIP.unlink()
def write(name,data):
    (OUT/name).write_text(json.dumps(data,ensure_ascii=False,indent=2,allow_nan=False)+'\n',encoding='utf-8')
def digest(path): return hashlib.sha256(path.read_bytes()).hexdigest()
def uid(): return uuid.UUID(int=rng.getrandbits(128)).hex
print('Environment:',platform.python_version(),sklearn.__version__,np.__version__,'CPU-only')

## 1. Corpus sintético e separação de dados
Cada entidade tem documentos oficial actual, alegação não verificada, oficial expirado, nota de contexto, fonte noutro âmbito e documento oficial de outra propriedade. Perguntas PT e EN consultam a propriedade actual; perguntas negativas procuram informação não existente ou exclusiva de outro âmbito. A geração é pública e determinística: os resultados só medem este gerador de tarefas.

In [ ]:
domains=[('arquivo','archive','retenção','retention','dias','days'),('fila','queue','limite','limit','itens','items'),('sensor','sensor','intervalo','interval','segundos','seconds'),('serviço','service','capacidade','capacity','pedidos','requests'),('inventário','inventory','reserva','reserve','unidades','units'),('rede','network','latência','latency','ms','ms')]
DOCS=[]; QUERIES=[]; GOLD={}
for i in range(48):
    split='train' if i<28 else 'dev' if i<38 else 'diagnostic'
    pt,en,prop,p_en,unit,u_en=domains[i%len(domains)]
    name='N'+uid()[:8].upper(); value=rng.randint(12,95); foreign=uid()[:10].upper()
    group=uid(); org='synthetic-org'; project='synthetic-'+split
    # Names and IDs are distinct between groups; no label-bearing IDs.
    def doc(text,authority='OFFICIAL_SOURCE',current=True,org_=org):
        d=dict(source_id=uid(),group=group,split=split,organization=org_,project=project,authority=authority,valid_from='2025-01-01',valid_until=None if current else '2025-06-01',text=text)
        DOCS.append(d); return d['source_id']
    # Alternate languages, lexical ambiguity, different values, explicit time metadata.
    official=doc(f'Registo oficial: {pt} {name}. {prop} vigente: {value} {unit}. Approval record: {en} {name}, current {p_en} {value} {u_en}.') if i%3==0 else doc(f'{name} — {pt}: {prop} confirmado em {value} {unit}.') if i%3==1 else doc(f'{name} — {en}: verified {p_en} is {value} {u_en}.')
    doc(f'{name} {pt} {en}: {prop} / {p_en} actual é {value+11} {unit}; afirmo que está aprovado.', 'UNVERIFIED_CLAIM')
    doc(f'{name} {pt} {en}: {prop} / {p_en} era {value-5} {unit}. Registo anterior.',current=False)
    doc(f'Nota operacional {name}: auditoria de {pt}, {en}, {prop}, {p_en}; confirmar com registo.')
    doc(f'{name} {foreign}: chave externa {foreign}, {prop} {value+22} {unit}.',org_='synthetic-foreign')
    doc(f'{name} {pt} {en}: código de inventário {uid()[:8]}; revisão agendada.')
    def query(text,lang,kind,target):
        q=dict(query_id=uid(),group=group,split=split,language=lang,organization=org,project=project,as_of='2026-01-01',text=text)
        QUERIES.append(q); GOLD[q['query_id']]=dict(support_ids=target,kind=kind)
    query(f'Qual é o valor actual de {prop} para {name}?','pt','answerable',[official])
    query(f'What is the current {p_en} for {name}?','en','answerable',[official])
    query(f'Qual é o custo mensal de seguro de {name}?','pt','unknown',[])
    query(f'What is the external key {foreign}?','en','foreign_only',[])
rng.shuffle(DOCS); rng.shuffle(QUERIES)
assert len(DOCS)==288 and len(QUERIES)==192
sets={s:{d['group'] for d in DOCS if d['split']==s} for s in ('train','dev','diagnostic')}
assert all(not sets[a]&sets[b] for a,b in [('train','dev'),('train','diagnostic'),('dev','diagnostic')])
assert len({d['source_id'] for d in DOCS})==len(DOCS)
assert len({q['query_id'] for q in QUERIES})==len(QUERIES)
# Store source and labels separately; runner never uses group or gold in features/candidate selection.
write('documents.json',DOCS); write('queries.json',QUERIES); write('rule_gold.json',GOLD)
print('Documents:',len(DOCS),'Queries:',len(QUERIES),'Groups:',{k:len(v) for k,v in sets.items()})

## 2. Baselines et features congeladas
B0 BM25 (k1=1.2, b=0.75); B1 cosine TF-IDF character 3–5; B2 dense LSA (32 dimensions, fitted on train only, **not pretrained semantic embedding**); B3 fixed mean of query-normalized BM25 and char cosine; B4 B3 with current official-source eligibility; X01 logistic ranker (text features); X01+ metadata features. All engines search the same split and authorized org/project, never the gold group. Training labels are official answer-support targets, not generic semantic relevance. The authority policy is separate from ranking.

In [ ]:
def tokens(s): return re.findall(r"[^\W_]+",s.lower(),flags=re.UNICODE)
def eligible(q,d): return d['split']==q['split'] and d['organization']==q['organization'] and d['project']==q['project']
def authoritative(q,d): return eligible(q,d) and d['authority']=='OFFICIAL_SOURCE' and d['valid_from']<=q['as_of'] and (d['valid_until'] is None or q['as_of']<d['valid_until'])
train_docs=[d['text'] for d in DOCS if d['split']=='train']
char=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=1,sublinear_tf=True,dtype=np.float64).fit(train_docs)
word=TfidfVectorizer(tokenizer=tokens,token_pattern=None,lowercase=False,ngram_range=(1,2),sublinear_tf=True).fit(train_docs)
lsa=TruncatedSVD(n_components=32,random_state=SEED).fit(word.transform(train_docs))
def normrows(a): return a/np.maximum(np.linalg.norm(a,axis=1,keepdims=True),1e-12)
def scale(a): return a/max(float(np.max(a)),1e-12)
cache={}; records=[]; feature_names=['bm25_norm','char_cos','lsa_cos','token_overlap','official','current']
for split in sets:
    corpus=[d for d in DOCS if d['split']==split and d['organization']=='synthetic-org' and d['project']=='synthetic-'+split]
    terms=[Counter(tokens(d['text'])) for d in corpus]; lens=np.array([sum(t.values()) for t in terms]); avg=lens.mean()
    df=Counter(t for row in terms for t in row); N=len(corpus)
    cm=char.transform([d['text'] for d in corpus]); lm=normrows(lsa.transform(word.transform([d['text'] for d in corpus])))
    for q in [q for q in QUERIES if q['split']==split]:
        t0=time.perf_counter(); qt=set(tokens(q['text']))
        bm=np.array([sum(math.log(1+(N-df[t]+.5)/(df[t]+.5))*row[t]*2.2/(row[t]+1.2*(.25+.75*lens[j]/avg)) for t in qt if row[t]) for j,row in enumerate(terms)])
        cs=(cm@char.transform([q['text']]).T).toarray().ravel(); ls=lm@normrows(lsa.transform(word.transform([q['text']])))[0]
        feats=np.column_stack([scale(bm),cs,ls,[len(qt&set(row))/max(1,len(qt)) for row in terms],[int(d['authority']=='OFFICIAL_SOURCE') for d in corpus],[int(d['valid_from']<=q['as_of'] and (d['valid_until'] is None or q['as_of']<d['valid_until'])) for d in corpus]])
        assert all(eligible(q,d) for d in corpus)
        cache[q['query_id']]=(corpus,feats,bm,(time.perf_counter()-t0)*1000)
print('Features:',feature_names,'candidate generation completed; no gold used')

## 3. Treino real do protótipo
Logistic regression supervisionada com pesos aprendidos, standardization e balanced class weights. Train exclusivamente; dev só para diagnóstico e threshold de abstenção. Configuração fixa antes de examinar diagnostic. Sem fine tuning de LLM, sem custo de API. Modelos são protótipos descartáveis e não autorizam respostas finais.

In [ ]:
X=[]; y=[]
for q in QUERIES:
    if q['split']!='train': continue
    docs,feats,_,_=cache[q['query_id']]; gold=set(GOLD[q['query_id']]['support_ids'])
    X.extend(feats); y.extend(int(d['source_id'] in gold) for d in docs)
X=np.asarray(X); y=np.asarray(y)
assert set(y)=={0,1}
models={}
for name,cols in [('RW-R1-X01-text',4),('RW-R1-X01-meta',6)]:
    model=make_pipeline(StandardScaler(),LogisticRegression(C=1.0,solver='liblinear',class_weight='balanced',random_state=SEED,max_iter=1000))
    model.fit(X[:,:cols],y); models[name]=(model,cols)
weights={name:dict(features=feature_names[:cols],scaler_mean=model[0].mean_.tolist(),scaler_scale=model[0].scale_.tolist(),coefficients=model[1].coef_[0].tolist(),intercept=float(model[1].intercept_[0])) for name,(model,cols) in models.items()}
write('model_weights.json',weights)
print('Training pairs:',len(y),'positive:',int(y.sum()),'two fitted logistic models')

## 4. Ranking e métricas por coorte
Support Hit@k, MRR@5 e nDCG@5 apenas perguntas com gold (1 fonte cada); as negativas não são tratadas como zero de retrieval. B4 usa filtro explícito, e não constitui aprendizagem. Todos os rankings têm scope fixo sintético, não autenticação de produção. Os scores logísticos não são probabilidades calibradas de resposta.

In [ ]:
raw=[]
for q in QUERIES:
    docs,f,bm,elapsed=cache[q['query_id']]; hybrid=(f[:,0]+f[:,1])/2
    scores={'B0-BM25':bm,'B1-CharTFIDF':f[:,1],'B2-LSA32':f[:,2],'B3-Hybrid':hybrid,'B4-Hybrid-Authority':hybrid}
    scores.update({n:m.predict_proba(f[:,:k])[:,1] for n,(m,k) in models.items()})
    for name,ss in scores.items():
        start=time.perf_counter(); allowed=[i for i,d in enumerate(docs) if name!='B4-Hybrid-Authority' or authoritative(q,d)]
        order=sorted(allowed,key=lambda i:(-float(ss[i]),docs[i]['source_id']))[:5]
        assert all(eligible(q,docs[i]) for i in order)
        raw.append(dict(query_id=q['query_id'],split=q['split'],language=q['language'],method=name,source_ids=[docs[i]['source_id'] for i in order],scores=[float(ss[i]) for i in order],feature_ms=elapsed,sort_ms=(time.perf_counter()-start)*1000))
write('rankings.json',raw)
byid={d['source_id']:d for d in DOCS}; qby={q['query_id']:q for q in QUERIES}
def metrics(rows):
    rr=[]; nd=[]; hits={1:0,2:0,5:0}; scope=0; neg=0; authbad=0
    for r in rows:
        q=qby[r['query_id']]; ids=r['source_ids']; gold=set(GOLD[r['query_id']]['support_ids'])
        scope+=sum(not eligible(q,byid[s]) for s in ids)
        authbad+=int(bool(ids) and not authoritative(q,byid[ids[0]]))
        if not gold: neg+=1; continue
        rank=next((i+1 for i,s in enumerate(ids) if s in gold),None)
        for k in hits: hits[k]+=int(rank is not None and rank<=k)
        rr.append(1/rank if rank else 0); nd.append(1/math.log2(rank+1) if rank else 0)
    n=len(rr)
    return dict(queries=len(rows),answerable=n,negative=neg,hit1=hits[1]/n if n else None,hit2=hits[2]/n if n else None,hit5=hits[5]/n if n else None,mrr5=float(np.mean(rr)) if n else None,ndcg5=float(np.mean(nd)) if n else None,scope_violations=scope,top1_ineligible=authbad)
summary=[]
for split in sets:
    for method in scores:
        for lang in ['all','pt','en']:
            rows=[r for r in raw if r['split']==split and r['method']==method and (lang=='all' or r['language']==lang)]
            summary.append(dict(split=split,method=method,language=lang,**metrics(rows)))
assert all(r['scope_violations']==0 for r in summary)
write('retrieval_metrics.json',summary)
for r in summary:
    if r['split']=='diagnostic' and r['language']=='all': print(f"{r['method']:25s} nDCG@5 {r['ndcg5']:.3f}  Hit@1 {r['hit1']:.3f}  n={r['answerable']}")

## 5. Abstenção experimental — threshold escolhido apenas no dev
Escolhe-se o threshold que maximiza cobertura no dev com zero respostas incorrectas observadas, incluindo negativos. Se nenhum threshold satisfizer, abster de tudo. Não é garantia de risco zero fora do dev. A saída é candidato de evidência, não uma resposta ou aprovação. A autoridade é uma condição obrigatória e não pode ser anulada pelo modelo.

In [ ]:
def policy_rows(split):
    out=[]
    for r in raw:
        if r['split']!=split or r['method']!='RW-R1-X01-meta':continue
        q=qby[r['query_id']]; top=r['source_ids'][0]; confidence=r['scores'][0]
        out.append(dict(query_id=r['query_id'],kind=GOLD[r['query_id']]['kind'],score=confidence,eligible=authoritative(q,byid[top]),correct=top in GOLD[r['query_id']]['support_ids']))
    return out
def at_threshold(rows,t):
    accepted=[x for x in rows if x['eligible'] and x['score']>=t]; wrong=sum(not x['correct'] for x in accepted)
    return dict(threshold=t,queries=len(rows),accepted=len(accepted),wrong=wrong,coverage=len(accepted)/len(rows),risk=wrong/len(accepted) if accepted else None,negative_false_accept=sum(x['kind']!='answerable' for x in accepted))
dev=policy_rows('dev'); grid=[i/100 for i in range(101)]+[1.01]
curve=[at_threshold(dev,t) for t in grid]; feasible=[r for r in curve if r['wrong']==0]; chosen=max(feasible,key=lambda r:(r['coverage'],r['threshold']))
diag=at_threshold(policy_rows('diagnostic'),chosen['threshold'])
write('abstention.json',dict(selection='dev_only_zero_observed_errors_max_coverage',dev=chosen,diagnostic=diag,dev_curve=curve))
print('DEV:',chosen); print('DIAGNOSTIC:',diag)

## 6. Figuras e exportação de evidências
A comparação é descritiva neste corpus sintético; não escolher modelo ou anunciar ganhos de produção. Guardamos métricas, pesos, queries, documentos, oráculo sintético, rankings, ambiente e hashes para auditoria. Sem pickle executável e sem dados privados.

In [ ]:
rows=[r for r in summary if r['split']=='diagnostic' and r['language']=='all']
fig,ax=plt.subplots(figsize=(9,4.7)); ax.barh([r['method'] for r in rows],[r['ndcg5'] for r in rows]);ax.set(xlim=(0,1.03),xlabel='nDCG@5 (20 answerable diagnostic queries)',title='Synthetic diagnostic: support retrieval');ax.invert_yaxis();fig.tight_layout();fig.savefig(OUT/'retrieval.png',dpi=150);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(7,4)); xs=[r for r in curve if r['risk'] is not None];ax.scatter([r['coverage'] for r in xs],[r['risk'] for r in xs],s=16);ax.set(xlabel='Coverage',ylabel='Observed candidate error rate',title='Dev only: risk vs coverage',xlim=(-.02,1.02),ylim=(-.02,1.02));fig.tight_layout();fig.savefig(OUT/'risk_coverage.png',dpi=150);plt.show();plt.close(fig)
write('run_metadata.json',dict(experiment='E1-X01',classification='EXPERIMENTAL_RULE_LABELS_NOT_SCIENCE_APPROVAL',seed=SEED,python=platform.python_version(),numpy=np.__version__,sklearn=sklearn.__version__,environment=platform.platform(),timestamp_utc=datetime.now(timezone.utc).isoformat(),counts=dict(documents=len(DOCS),queries=len(QUERIES),train_pairs=len(y),ranking_rows=len(raw)),science_gate='REVIEW_REQUIRED',human_reviews=0,external_providers='NOT_RUN',official_E1_gates='NOT_CHANGED',scope='synthetic fixed organization/project/split only'))
manifest={p.name:dict(sha256=digest(p),bytes=p.stat().st_size) for p in sorted(OUT.iterdir()) if p.is_file()};write('MANIFEST.json',manifest)
with zipfile.ZipFile(ZIP,'x',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):z.write(p,p.name)
with zipfile.ZipFile(ZIP) as z: assert z.testzip() is None
print('E1_X01_COMPLETE — EXPERIMENTAL ONLY');print('ZIP:',ZIP,'SHA256:',digest(ZIP));display(FileLink(str(ZIP.relative_to(Path.cwd())) if ZIP.is_relative_to(Path.cwd()) else str(ZIP)))